In [1]:
library(report)
library(lme4)
library(lmerTest)
library(BayesFactor)
library(tseries)
library(dplyr)
library(Kendall)
source('bayesMannWhitney.R')

Loading required package: Matrix


Attaching package: ‘lmerTest’


The following object is masked from ‘package:lme4’:

    lmer


The following object is masked from ‘package:stats’:

    step


Loading required package: coda

************
Welcome to BayesFactor 0.9.12-4.4. If you have questions, please contact Richard Morey (richarddmorey@gmail.com).

Type BFManual() to open the manual.
************

Registered S3 method overwritten by 'quantmod':
  method            from
  as.zoo.data.frame zoo 


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [6]:
df <- read.csv('processed/questionnaires.csv')
df$interface <- factor(df$interface, levels=c('listwise-ranking', 'pointwise', 'pairwise', 'listwise-rating', 'listwise-notie', 'listwise-cantie'))
df

confidence,confidenceTimestamp,expectation,expectationTimestamp,feedback,finishTimestamp,improvement,improvementTimestamp,incompletePreference,incompletePreferenceTimestamp,⋯,regretTimestamp,satisfaction,satisfactionTimestamp,sessionID,studyID,taskID,userID,interface,domain,agg_satisfaction
<int>,<dbl>,<int>,<dbl>,<chr>,<dbl>,<int>,<dbl>,<int>,<dbl>,⋯,<dbl>,<int>,<dbl>,<chr>,<chr>,<int>,<chr>,<fct>,<chr>,<dbl>
27,1.661894e+12,80,1.661894e+12,"The AI seems pretty unresponsive once you show some level of satisfaction with the text. I don't feel this is the most efficient way to work with it and it would be better if there were a few more detailed scales to give feedback on, instead of just liking or disliking.",1.661894e+12,44,1.661894e+12,50,1.661894e+12,⋯,1.661894e+12,76,1.661894e+12,S169,630e6ecbbd2508c3ffff37d5,1,P095,pointwise,text,56.75
68,1.661894e+12,94,1.661894e+12,"This one was great from the first try, so there was nothing to improve.",1.661894e+12,34,1.661894e+12,35,1.661894e+12,⋯,1.661894e+12,94,1.661894e+12,S169,630e6ecbbd2508c3ffff37d5,2,P095,pointwise,text,72.50
31,1.661895e+12,60,1.661895e+12,Feels like it didn't listen to the feedback at all.,1.661895e+12,30,1.661895e+12,23,1.661895e+12,⋯,1.661895e+12,28,1.661895e+12,S169,630e6ecbbd2508c3ffff37d5,3,P095,pointwise,text,37.25
34,1.661898e+12,100,1.661898e+12,"AI seems to 'catch the point', it felt like AI analyzed feedback and tried to fix its texts, but one time it had an issue and cut the sentence in the middle.",1.661898e+12,95,1.661898e+12,0,1.661898e+12,⋯,1.661898e+12,100,1.661898e+12,S058,630e6ecbbd2508c3ffff37d5,2,P097,pointwise,text,82.25
27,1.661899e+12,59,1.661899e+12,I think AI included too little information for such detailed text.,1.661899e+12,40,1.661899e+12,0,1.661899e+12,⋯,1.661899e+12,40,1.661899e+12,S058,630e6ecbbd2508c3ffff37d5,3,P097,pointwise,text,41.50
39,1.661899e+12,80,1.661899e+12,AI seemed to learn and try to include information given in feedback.,1.661899e+12,76,1.661899e+12,0,1.661899e+12,⋯,1.661899e+12,69,1.661899e+12,S058,630e6ecbbd2508c3ffff37d5,1,P097,pointwise,text,66.00
24,1.661927e+12,38,1.661927e+12,i broke it,1.661927e+12,0,1.661927e+12,26,1.661927e+12,⋯,1.661927e+12,51,1.661927e+12,S083,630e6ecbbd2508c3ffff37d5,2,P279,pointwise,text,28.25
74,1.661927e+12,80,1.661927e+12,,1.661928e+12,0,1.661928e+12,100,1.661928e+12,⋯,1.661928e+12,61,1.661927e+12,S083,630e6ecbbd2508c3ffff37d5,1,P279,pointwise,text,53.75
0,1.661927e+12,23,1.661927e+12,"With my feedback, I tried to influence the addition of information about the vice-president and Raku, but at some point al stopped responding and did not change the text anymore. Up to a point it was better. In my opinion, al selects irrelevant information to summarize.",1.661927e+12,26,1.661927e+12,21,1.661927e+12,⋯,1.661927e+12,12,1.661927e+12,S310,630e6ecbbd2508c3ffff37d5,1,P238,pointwise,text,15.25


In [23]:
# 'satisfaction', 'confidence', 'expectation', 'improvement', 'incompletePreference', 'regret'
m1 <- lmer(incompletePreference ~ interface + (1|domain) + (1|taskID) + (1|userID), data=df)
summary(m1)
print(report(m1), verbose=TRUE)

boundary (singular) fit: see help('isSingular')



Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: incompletePreference ~ interface + (1 | domain) + (1 | taskID) +  
    (1 | userID)
   Data: df

REML criterion at convergence: 5097

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-2.9151 -0.4203 -0.0892  0.4460  3.2647 

Random effects:
 Groups   Name        Variance Std.Dev.
 userID   (Intercept) 436.6    20.89   
 taskID   (Intercept)   0.0     0.00   
 domain   (Intercept)   0.0     0.00   
 Residual             338.2    18.39   
Number of obs: 557, groups:  userID, 209; taskID, 3; domain, 2

Fixed effects:
                         Estimate Std. Error      df t value Pr(>|t|)    
(Intercept)                20.489      4.108 199.153   4.988 1.33e-06 ***
interfacepointwise          7.404      5.868 201.162   1.262 0.208514    
interfacepairwise          21.021      5.644 198.234   3.724 0.000255 ***
interfacelistwise-rating   17.178      5.806 194.220   2.959 0.003475 ** 
inte

boundary (singular) fit: see help('isSingular')



Random effect variances not available. Returned R2 does not account for random effects.


boundary (singular) fit: see help('isSingular')



Random effect variances not available. Returned R2 does not account for random effects.
We fitted a linear mixed model (estimated using REML and nloptwrap optimizer)
to predict incompletePreference with interface
 (formula: incompletePreference ~ interface). The model included domain, taskID
and userID
 as random effects (formula: list(~1 | domain, ~1 | taskID, ~1 | userID)). The
model's explanatory power related to the fixed effects alone (marginal R2) is
0.16. The model's intercept, corresponding to interface = listwise-ranking
, is at 20.49 (95% CI [12.42, 28.56], t(547) = 4.99, p < .001). Within this
model:

  - The effect of interface [pointwise] is statistically non-significant and
positive (beta = 7.40, 95% CI [-4.12, 18.93], t(547) = 1.26, p = 0.208; Std.
beta = 0.26, 95% CI [-0.14, 0.66])
  - The effect of interface [pairwise] is statistically significant and positive
(beta = 21.02, 95% CI [9.93, 32.11], t(547) = 3.72, p < .001; Std. beta = 0.73,
95% CI [0.34, 1.11])
  - The e